# Mutation-Conditioned GraphSWARM

Проверяем, улучшает ли **пересчёт сообщений после каждого обновления скрытых
состояний** прогноз ΔΔG по сравнению с однократно вычисленным контекстом.
Используем замороженные представления остатков ProteinMPNN/ThermoMPNN размерности
384, информацию о замене вводим в мутируемом остатке. Остатки взаимодействуют
по C-alpha kNN-графу WT-структуры.

`graph_static` один раз вычисляет обусловленные мутацией сообщения и повторно
использует их; `graph_swarm` пересчитывает сообщения на каждом шаге.
Контроли — `mutation_self` и `graph_swarm_t1`; основное сравнение — **static vs swarm**:

$$D = \mathrm{MacroMAE}_{static} - \mathrm{MacroMAE}_{swarm}.$$

**D > 0 означает преимущество GraphSWARM.** MacroMAE — среднее MAE по белкам,
с равным весом каждого белка; единицы — ккал/моль.

Скрытые итерации — вычислительные раунды, не физическое время. На малых
белковых графах **T=4 не гарантирует локального охвата**: информация может
достигнуть всего графа. Анализ validation — исследовательский (exploratory).
`head_holdout` и `legacy_test` здесь защищены и не оцениваются.


## 2. Настройки владельца

По умолчанию **NON-SCIENTIFIC ENGINEERING CHECK**: 4 train-белка и 2 validation-белка,
штатный smoke Stage 3 — 2 эпохи, seed 42, только `graph_static` и `graph_swarm`.
Точность smoke нельзя интерпретировать научно.

`full` использует все **209 train / 31 validation**, четыре архитектуры,
40 эпох и seeds 42–51. Для него обязательно `CONFIRM_FULL_RUN = True`.
Научные параметры зафиксированы в библиотеке; переопределений здесь нет.
Новый запуск требует нового `RUN_ID`; `RESUME = True` продолжает совместимый запуск.


In [ ]:
import torch

RUN_MODE = "smoke"
RUN_ID = "graph_swarm_v1_smoke_01"
RESUME = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIRM_FULL_RUN = False
RUN_LATENT_DIAGNOSTIC = False

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError('RUN_MODE должен быть "smoke" или "full".')
if RUN_MODE == "full" and CONFIRM_FULL_RUN is not True:
    raise ValueError("Для полного обучения установите CONFIRM_FULL_RUN = True и новый RUN_ID.")


## 3. Пути и окружение

Запускайте из корня репозитория или `notebooks/` в окружении NeuroPP.
Нужны локальная ThermoMPNN, MegaScale и существующий кэш признаков.
Помощник ниже извлекает только нужный файл из уже локального ZIP, если файл
отсутствует. CSV понадобится впервые в разделе загрузки меток; скачиваний нет.


In [ ]:
import gc
import json
import platform
import sys
import time
from pathlib import Path
from shutil import copyfileobj
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (path / "src/neuropp/training.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден корень Neuro_Project_Pilot; откройте репозиторий или notebooks/.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

THERMOMPNN_DIR = PROJECT_ROOT / "external/ThermoMPNN"
DATA_DIR = PROJECT_ROOT / "data/megascale"
FEATURE_DIR = PROJECT_ROOT / "data/features/thermompnn"
GRAPH_DIR = PROJECT_ROOT / "data/graphs/graph_swarm_v1"
SPLIT_PATH = PROJECT_ROOT / "splits/graph_swarm_v1.json"
PROTOCOL_PATH = PROJECT_ROOT / "experiments/graph_swarm_v1/protocol.json"
CSV_PATH = DATA_DIR / "Processed_K50_dG_datasets/Tsuboyama2023_Dataset2_Dataset3_20230416.csv"
PDB_DIR = DATA_DIR / "AlphaFold_model_PDBs"

def ensure_local_file(path, zip_path):
    if path.is_file():
        return path
    if not zip_path.is_file():
        raise FileNotFoundError(f"Нет {path} и локального архива {zip_path}; подготовьте их вручную.")
    with ZipFile(zip_path) as archive:
        matches = [item for item in archive.infolist()
                   if not item.is_dir() and Path(item.filename).name == path.name]
        if len(matches) != 1:
            raise FileNotFoundError(f"В {zip_path} нет единственного файла {path.name}.")
        path.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(matches[0]) as source, path.open("wb") as destination:
            copyfileobj(source, destination)
    return path

print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | device={DEVICE}")
print(f"RUN_MODE={RUN_MODE} | RUN_ID={RUN_ID}")


## 4. Замороженный протокол и происхождение признаков

Проверяем протокол, разбиение и локальные исходные веса. `load_frozen_core`
нужен только для аудита и подписи существующего кэша: модель не вызывается,
голова стабильности ThermoMPNN не используется. После аудита освобождаем модель.


In [ ]:
from neuropp.models import MutationGraphModel, VARIANTS, architecture_specification
from neuropp.protocol import COUNTS, GraphParameters
from neuropp.provenance import load_frozen_core
from neuropp.splits import file_hash, validate_split
from neuropp.training import TrainingProtocol

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
split_manifest = validate_split(json.loads(SPLIT_PATH.read_text(encoding="utf-8")))
split_manifest_sha256 = file_hash(SPLIT_PATH)
assert protocol["training"] == TrainingProtocol().as_dict(), "Полный протокол изменён."
assert protocol["model"] == architecture_specification(), "Спецификация моделей изменена."
assert protocol["graph_parameters"] == GraphParameters().as_dict(), "Протокол графа изменён."
assert protocol["splits"]["counts"] == split_manifest["counts"] == COUNTS

core, _, provenance = load_frozen_core(THERMOMPNN_DIR)
assert provenance["status"] == "passed", "Аудит происхождения признаков не пройден."
assert provenance["checks"]["proteinmpnn_tensor_match"]["passed"], "Веса ProteinMPNN не совпадают."
assert provenance["checks"]["feature_path"]["dimension"] == protocol["data"]["feature_dimension"] == 384
feature_signature = provenance["feature_signature"]
del core
gc.collect()
if torch.device(DEVICE).type == "cuda":
    torch.cuda.empty_cache()

print("Feature dimension = 384 | feature provenance passed | ProteinMPNN tensor match passed")
print("Split counts (train / validation / head_holdout / legacy_test) = 209 / 31 / 30 / 28")
print("head_holdout protected | legacy_test protected")


## 5. Только development-белки

Smoke берёт первые 4 train и первые 2 validation ID в лексикографическом
порядке, независимо от меток и качества моделей. Full берёт все development ID
из замороженного manifest. Защищённые белки не включаются даже в records.


In [ ]:
train_ids = sorted(split_manifest["train"])
validation_ids = sorted(split_manifest["validation"])
if RUN_MODE == "smoke":
    train_ids, validation_ids = train_ids[:4], validation_ids[:2]
else:
    assert (len(train_ids), len(validation_ids)) == (209, 31)
development_ids = train_ids + validation_ids
assert not set(development_ids) & set(split_manifest["head_holdout"])
assert not set(development_ids) & set(split_manifest["legacy_test"])
print(f"Train proteins: {len(train_ids)} | validation proteins: {len(validation_ids)}")
if RUN_MODE == "smoke":
    print("Train IDs:", train_ids)
    print("Validation IDs:", validation_ids)


## 6. WT-структуры, признаки и графы

Stage 1 загружает WT PDB, сверяет существующий кэш **L × 384** по проверенной
подписи и создаёт `ProteinRecord` с замороженным графом. Отсутствующий кэш
признаков вызывает ошибку; признаки не пересчитываются. Графы проходят
штатный `GraphCache`. Старые baseline-прогнозы не используются как входы.


In [ ]:
from neuropp.cache import GraphCache
from neuropp.graph import ProteinRecord
from neuropp.local_artifacts import load_existing_record, pdb_path_for

graph_cache = GraphCache(GRAPH_DIR)
records: dict[str, ProteinRecord] = {}
for protein_id in development_ids:
    pdb_path = ensure_local_file(pdb_path_for(PDB_DIR, protein_id), DATA_DIR / "AlphaFold_model_PDBs.zip")
    record, _ = load_existing_record(protein_id, pdb_path, FEATURE_DIR, feature_signature)
    graph_cache.get_or_build(record)
    records[protein_id] = record
lengths = np.array([len(record.sequence) for record in records.values()])
display(pd.DataFrame([{"proteins": len(records), "total residues": int(lengths.sum()),
                       "length min": int(lengths.min()), "length median": float(np.median(lengths)),
                       "length max": int(lengths.max())}]))


## 7. Метки train + validation

**Здесь впервые открывается реальный источник меток при ручном запуске владельцем.**
Используем только защищённый `DevelopmentDataAccess.load_megascale_csv` для
выбранных ID. Он сохраняет порядок валидных одиночных замен и проверяет
согласование WT/мутации. Цель **ΔΔG = −ddG_ML**, положительное значение означает
дестабилизацию; единицы — ккал/моль. Индивидуальные метки не выводим.


In [ ]:
from neuropp.data_access import DevelopmentDataAccess
from neuropp.training import DevelopmentDataset, TrainingProtein

access = DevelopmentDataAccess(split_manifest)
access.require_development_ids(development_ids)
ensure_local_file(CSV_PATH, DATA_DIR / "Processed_K50_dG_datasets.zip")
labels = access.load_megascale_csv(development_ids, CSV_PATH, records)
dataset = DevelopmentDataset(
    train={pid: TrainingProtein(records[pid], labels[pid], access) for pid in train_ids},
    validation={pid: TrainingProtein(records[pid], labels[pid], access) for pid in validation_ids},
    access=access, split_manifest_sha256=split_manifest_sha256, feature_signature=feature_signature,
)
dataset.validate(RUN_MODE)

label_rows = []
for split, proteins in (("train", dataset.train), ("validation", dataset.validation)):
    counts = np.array([len(protein.labels.queries) for protein in proteins.values()])
    label_rows.append({"split": split, "proteins": len(proteins), "mutations": int(counts.sum()),
                       "mutations/protein min": int(counts.min()),
                       "mutations/protein median": float(np.median(counts)),
                       "mutations/protein max": int(counts.max())})
display(pd.DataFrame(label_rows).set_index("split"))


## 8. Четыре модели

- **Self**: нет обмена между узлами внутри рекуррентных обновлений;
  итоговый readout всё же использует среднее по остаткам.
- **Static**: один обусловленный мутацией набор сообщений повторно используется.
- **Swarm**: сообщения пересчитываются на каждом шаге.
- **Swarm T1**: один раунд взаимодействия.

Таблица ниже создаёт модели только для подсчёта параметров, без forward и обучения.
У Self зарегистрированная attention-сеть не участвует в вычислениях.


In [ ]:
model_rows = []
for architecture in VARIANTS:
    model = MutationGraphModel(architecture)
    counts = model.parameter_counts()
    assert counts["total_registered"] == 88_033
    assert counts["effectively_used"] == (83_329 if architecture == "mutation_self" else 88_033)
    model_rows.append({"architecture": architecture, "steps": model.config.steps,
                       "total parameters": counts["total_registered"],
                       "effectively used parameters": counts["effectively_used"]})
del model
display(pd.DataFrame(model_rows).set_index("architecture"))


## 9. LONG-RUN CELL — обучение

**Следующая ячейка запускает обучение. Codex её не запускал.**

`run_experiment` управляет обучением, выбором лучшей эпохи по validation
protein macro MAE, checkpoint/resume и сохранением таблиц. Smoke использует
штатные defaults Stage 3; full — замороженный `TrainingProtocol` без overrides.
Результаты попадут в `results/graph_swarm_v1/<RUN_ID>/`, веса —
в `weights/graph_swarm_v1/<RUN_ID>/`. Ошибки не подавляются.


In [ ]:
from neuropp.training import run_experiment

if RUN_MODE == "full" and CONFIRM_FULL_RUN is not True:
    raise ValueError("Полное обучение требует CONFIRM_FULL_RUN = True.")
started = time.perf_counter()
result = run_experiment(
    dataset=dataset, mode=RUN_MODE, device=DEVICE, run_id=RUN_ID,
    output_root=PROJECT_ROOT, resume=RESUME,
)
elapsed = time.perf_counter() - started
full_run = result.manifest["mode"] == "full"
print("Results directory:", result.results_directory)
print("Weights directory:", result.weights_directory)
print(f"Elapsed wall-clock time: {elapsed:.1f} s ({elapsed / 3600:.2f} h)")
if not full_run:
    print("NON-SCIENTIFIC SMOKE — do not interpret accuracy.")


## 10. История обучения

Одна кривая validation protein macro MAE на архитектуру: в full усредняем по seeds,
в smoke показываем доступные кривые. Ниже — выбранная лучшая эпоха каждого запуска.
Все таблицы берём из `ExperimentResult`; дополнительные модели не оцениваем.


In [ ]:
history = pd.DataFrame(result.history)
validation_summary = pd.DataFrame(result.validation_summary)
trained_architectures = [name for name in VARIANTS
                         if name in result.manifest["training_protocol"]["variants"]]
fig, ax = plt.subplots(figsize=(8, 4))
for architecture in trained_architectures:
    rows = history.loc[history["variant"] == architecture]
    if full_run:
        curve = rows.groupby("epoch")["protein_macro_MAE"].mean()
        ax.plot(curve.index, curve.values, label=architecture)
    else:
        for seed, curve in rows.groupby("seed"):
            ax.plot(curve["epoch"], curve["protein_macro_MAE"], label=f"{architecture}, seed {seed}")
ax.set(xlabel="Эпоха", ylabel="Validation protein_macro_MAE (ккал/моль)",
       title="Среднее по seeds" if full_run else "NON-SCIENTIFIC SMOKE")
if not full_run:
    ax.set_xticks(sorted(history["epoch"].unique()))
ax.legend()
fig.tight_layout()
plt.show()
display(validation_summary.pivot(index="seed", columns="variant", values="best_epoch")
        .reindex(columns=trained_architectures).rename_axis(columns="architecture"))


## 11. Validation: сводка моделей

Full показывает среднее и выборочное стандартное отклонение macro MAE по seeds,
средние pooled MAE/RMSE и лучшую эпоху. Pooled-метрики дают одинаковый вес каждой
мутации. В smoke — только значения одного seed, без оценки межseed-разброса;
`mutation_self` и `graph_swarm_t1` в штатном smoke не обучаются.


In [ ]:
if full_run:
    model_summary = validation_summary.groupby("variant").agg(
        protein_macro_MAE_mean=("protein_macro_MAE", "mean"),
        protein_macro_MAE_std=("protein_macro_MAE", "std"),
        pooled_MAE_mean=("pooled_MAE", "mean"),
        pooled_RMSE_mean=("pooled_RMSE", "mean"),
        best_epoch_mean=("best_epoch", "mean"),
    ).reindex(VARIANTS)
else:
    print("NON-SCIENTIFIC SMOKE: значения одного seed; std не оценивается.")
    model_summary = validation_summary.set_index("variant")[
        ["seed", "protein_macro_MAE", "pooled_MAE", "pooled_RMSE", "best_epoch"]
    ].reindex(trained_architectures)
display(model_summary.rename_axis("architecture").round(6))


## 12. Главное сравнение: graph_static vs graph_swarm

Для каждого seed показываем $D = \mathrm{MacroMAE}_{static} - \mathrm{MacroMAE}_{swarm}$.
Full читает **crossed bootstrap seeds × proteins** из результата Stage 3:
10 000 повторов, 95% percentile-интервал. Другую статистическую процедуру
не пересчитываем. Численные совпадения определяются порогом $10^{-6}$.

Интервал выше нуля и не менее 7 положительных seeds означают сильную
**исследовательскую поддержку на validation**. Положительный эффект с интервалом,
содержащим ноль, остаётся неубедительным; близкий к нулю или нестабильный эффект
не даёт ясного преимущества. Отрицательный эффект означает отсутствие улучшения.
**Ни один исход validation не является окончательным подтверждением.**


In [ ]:
validation_effects = pd.DataFrame(result.validation_effects)
display(validation_effects[["seed", "static_macro_MAE", "swarm_macro_MAE", "static_minus_swarm"]]
        .rename(columns={"static_minus_swarm": "D = static - swarm"}).set_index("seed").round(6))
if full_run:
    bootstrap = result.validation_bootstrap
    effects = bootstrap["seed_effect_summary"]
    mean_D = effects["mean_effect"]
    low, high = bootstrap["CI95_low"], bootstrap["CI95_high"]
    display(pd.Series({
        "mean D": mean_D, "std D": effects["standard_deviation"],
        "positive seeds": effects["positive_seed_count"],
        "negative seeds": effects["negative_seed_count"],
        "tied seeds": effects["tied_seed_count"],
        "crossed bootstrap CI95_low": low, "crossed bootstrap CI95_high": high,
    }).to_frame("Значение"))
    if low > 0 and effects["positive_seed_count"] >= 7:
        interpretation = "strong exploratory validation support"
    elif abs(mean_D) <= 1e-6:
        interpretation = "no clear validation advantage"
    elif mean_D < 0:
        interpretation = "GraphSWARM does not improve over GraphStatic on validation"
    elif mean_D > 0 and low <= 0 <= high:
        interpretation = "positive but inconclusive exploratory validation result"
    else:
        interpretation = "no clear validation advantage"
    print(f"{interpretation}; mean D = {mean_D:.6g} ккал/моль.")
else:
    print("NON-SCIENTIFIC SMOKE — do not interpret accuracy.")
    print("D выше служит только проверкой таблиц; научный вывод и bootstrap-интервал не интерпретируются.")


## 13. Распределение эффекта по validation-белкам

Из `validation_per_protein` сначала усредняем MAE каждого белка по seeds,
затем считаем $gain_p = \mathrm{MAE}_{static,p} - \mathrm{MAE}_{swarm,p}$.
Положительный gain означает преимущество GraphSWARM; $|gain| \leq 10^{-6}$ — ничья.
Это исследовательский срез validation; в smoke он проверяет только отображение.


In [ ]:
validation_per_protein = pd.DataFrame(result.validation_per_protein)
protein_mae = (validation_per_protein.loc[
    validation_per_protein["variant"].isin(["graph_static", "graph_swarm"])
].groupby(["protein_id", "variant"])["MAE"].mean().unstack("variant"))
gain = protein_mae["graph_static"] - protein_mae["graph_swarm"]
assert set(gain.index) == set(dataset.validation) and gain.notna().all()
display(pd.Series({"validation proteins": len(gain),
                   "GraphSWARM better": int((gain > 1e-6).sum()),
                   "GraphStatic better": int((gain < -1e-6).sum()),
                   "tied": int((gain.abs() <= 1e-6).sum()),
                   "median gain": float(gain.median())}).to_frame("Значение"))
fig, ax = plt.subplots(figsize=(8, 3))
ordered_gain = gain.sort_values()
ax.plot(np.arange(1, len(gain) + 1), ordered_gain.values, "o-", markersize=4)
ax.axhline(0, color="black", linewidth=1)
ax.set(xlabel="Ранг validation-белка по gain", ylabel="MAE static − MAE swarm (ккал/моль)",
       title="Exploratory validation" if full_run else "NON-SCIENTIFIC SMOKE")
fig.tight_layout()
plt.show()


## 14. Необязательная диагностика: latent representation response

По умолчанию выключена (`RUN_LATENT_DIAGNOSTIC = False`); ячейка ничего не делает.
При включении берём лексикографически первый **уже загруженный validation-белок**,
первую валидную мутацию в порядке CSV и первый seed протокола. Выбор не зависит
от ошибок или преимущества модели. Загружаем его лучший GraphSWARM checkpoint
через проверенный API Stage 3, используя сохранённую конфигурацию.

Сравниваем введение мутации с согласованным `inject_mutation=False` reference
(тот же query остаётся в readout):
$\delta_i^t = \|h_{i,injected}^t - h_{i,reference}^t\|_2$, $t=0,\ldots,T$.
На рисунке — отклик скрытого представления по позиции WT и вычислительному раунду.
Это **latent representation response**, не распространение энергии,
конформационная волна, физическое время или молекулярная динамика.


In [ ]:
if RUN_LATENT_DIAGNOSTIC:
    from neuropp.training import load_checkpoint, prepare_protein

    assert result.manifest["completed"], "Сначала завершите основной запуск."
    diagnostic_id = sorted(dataset.validation)[0]
    diagnostic_seed = result.manifest["training_protocol"]["seeds"][0]
    checkpoint_path = result.weights_directory / f"graph_swarm_seed{diagnostic_seed}_best.pt"
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    configuration = payload["model_configuration"]
    diagnostic_model = MutationGraphModel(configuration["variant"], steps=configuration["steps"])
    diagnostic_model = diagnostic_model.to(device="cpu", dtype=torch.float32)
    expected_metadata = {key: result.manifest[key] for key in (
        "run_id", "mode", "scientific_status", "training_protocol", "split_manifest_sha256",
        "feature_signature", "dataset_signature", "pytorch_version", "device_description", "torch_num_threads",
    )}
    expected_metadata.update(variant="graph_swarm", seed=diagnostic_seed,
                             initialization_hash=result.manifest["initialization_hashes"][str(diagnostic_seed)])
    load_checkpoint(checkpoint_path, diagnostic_model, None, expected_metadata, kind="best")
    diagnostic_model.eval()
    tensors = prepare_protein(dataset.validation[diagnostic_id], device="cpu")
    inputs = tensors.forward_inputs(0, 1)
    with torch.inference_mode():
        _, injected = diagnostic_model(*inputs, return_diagnostics=True, inject_mutation=True)
        _, reference = diagnostic_model(*inputs, return_diagnostics=True, inject_mutation=False)
        delta = torch.stack([torch.linalg.vector_norm(left[0] - right[0], dim=-1)
                             for left, right in zip(injected.hidden_states, reference.hidden_states)]).cpu().numpy()
    fig, ax = plt.subplots(figsize=(8, 3))
    image = ax.imshow(delta, origin="lower", aspect="auto", interpolation="nearest")
    ax.axvline(int(injected.mutation_rows[0]), color="white", linestyle="--", linewidth=1)
    ax.set(xlabel="Позиция WT (индекс с 0)", ylabel="Вычислительный раунд t",
           title=f"latent representation response — {diagnostic_id}, seed {diagnostic_seed}")
    ax.set_yticks(np.arange(delta.shape[0]))
    fig.colorbar(image, ax=ax, label="Норма разности скрытых состояний")
    fig.tight_layout()
    plt.show()
    del diagnostic_model, payload, tensors, inputs, injected, reference


## 15. Граница этого эксперимента

Тетрадка заканчивается на **train + validation**. Она не обращается к меткам
`head_holdout` и `legacy_test` и не даёт окончательного подтверждающего вывода.

После **FULL** результатов:

1. Передать validation-результаты для интерпретации.
2. Решить, зафиксированы ли архитектура и протокол.
3. Записать хэши зафиксированного кода и checkpoints.
4. Только затем создать отдельный этап оценки защищённого holdout.

Кода разблокировки holdout в этой тетрадке нет.
